In [4]:
NUPLAN_DATA_PATH="/home/taimor/data1/nuplan-v1.1/splits/mini" # nuplan training data path (e.g., "/data/nuplan-v1.1/trainval")
NUPLAN_MAP_PATH="/home/taimor/data1/maps" # nuplan map path (e.g., "/data/nuplan-v1.1/maps")

TRAIN_SET_PATH="/home/taimor/data1/preprocess" # preprocess training data
###################################



process raw data

In [9]:
import sys
import os
import subprocess

# --- 1. SET YOUR REPO PATH ---
# Change this to the absolute path of your Diffusion-Planner repository
diffusion_planner_path = "/home/taimor/V0.1-diffusion-es/Diffusion-Planner/"
# -----------------------------------


# --- 2. DEFINE YOUR PATHS AND ARGS ---
# These are the values you provided
data_path = "/home/taimor/data1/nuplan-v1.1/splits/mini"
map_path = "/home/taimor/data1/maps"
save_path = "/home/taimor/data1/preprocess"
total_scenarios = "10" # Arguments must be strings
# -------------------------------------


# --- 3. PREPARE AND RUN THE COMMAND ---
script_name = "data_process.py"
script_path = os.path.join(diffusion_planner_path, script_name)

# Check if the script exists
if not os.path.exists(script_path):
    print(f"ERROR: Script not found at '{script_path}'")
    print("Please make sure the 'diffusion_planner_path' variable is set correctly.")
else:
    # Build the full command with all arguments
    command_list = [
        sys.executable,  # The path to your 'nuplan' kernel's python
        script_name,
        "--data_path", data_path,
        "--map_path", map_path,
        "--save_path", save_path,
        "--total_scenarios", total_scenarios
    ]

    print("--- Running Command ---")
    # Print the command in a readable format
    print(" ".join(command_list))
    print("-------------------------")
    print("Starting data processing... this may take a while.")
    
    # Run the command from the repository's directory
    process = subprocess.Popen(
        command_list,
        cwd=diffusion_planner_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True
    )

    # Print the output in real-time
    for line in process.stdout:
        print(line, end='')

    process.wait()

    print("-------------------------")
    if process.returncode == 0:
        print("✅ Script finished successfully!")
    else:
        print(f"❌ Script failed with return code: {process.returncode}")

--- Running Command ---
/home/taimor/miniconda/envs/nuplan/bin/python data_process.py --data_path /home/taimor/data1/nuplan-v1.1/splits/mini --map_path /home/taimor/data1/maps --save_path /home/taimor/data1/preprocess --total_scenarios 10
-------------------------
Starting data processing... this may take a while.
None
Total number of scenarios: 10

  0%|          | 0/10 [00:00<?, ?it/s]/home/taimor/V0.1-diffusion-es/nuplan-devkit/nuplan/common/maps/nuplan_map/utils.py:413: RuntimeWarning: invalid value encountered in cast
  return elements.iloc[np.where(elements[column_label].to_numpy().astype(int) == int(desired_value))]
/home/taimor/V0.1-diffusion-es/nuplan-devkit/nuplan/common/maps/nuplan_map/utils.py:413: RuntimeWarning: invalid value encountered in cast
  return elements.iloc[np.where(elements[column_label].to_numpy().astype(int) == int(desired_value))]

 10%|█         | 1/10 [00:02<00:19,  2.13s/it]/home/taimor/V0.1-diffusion-es/nuplan-devkit/nuplan/common/maps/nuplan_map/utils.

#Great! You've successfully completed the data preprocessing step.

#Here’s a breakdown of what you have and what to do next.

#What Are Those 27 .npz Files?
#They are your training data. The data_process.py script took your 10 raw nuPlan scenarios (from the minisplit) and converted them into a format that the model can understand.

#.npz is a standard NumPy file format that stores multiple data arrays (tensors) in a single compressed file.

#Each of your 27 files is one training sample. The script likely generated multiple samples from each of the 10 scenarios (e.g., by taking different 8-second clips), which is why you have more files than scenarios.

#Inside each file are the numerical arrays for map data, agent trajectories, the ego vehicle's path, etc., all ready to be fed into the neural network.

In [12]:
import os
import glob
import json

# Path to your preprocessed .npz files
preprocess_path = "/home/taimor/data1/preprocess"

# The name of the new JSON file we will create
output_json_name = "minisplit_train_list.json"
output_json_path = os.path.join(preprocess_path, output_json_name)

# Use glob to find all files ending in .npz in that directory
# We only want the filename (e.g., "sg-one-north_...npz"), not the full path
npz_files = [os.path.basename(f) for f in glob.glob(os.path.join(preprocess_path, "*.npz"))]

if not npz_files:
    print(f"ERROR: No .npz files found in '{preprocess_path}'")
else:
    print(f"Found {len(npz_files)} .npz files.")
    
    # Write this list of filenames to the new JSON file
    with open(output_json_path, 'w') as f:
        json.dump(npz_files, f, indent=4)
        
    print(f"✅ Successfully created list file at: {output_json_path}")
    print("\n--- JSON file content (first 5 files) ---")
    print(json.dumps(npz_files[:5], indent=2))
    print("------------------------------------------")

Found 27 .npz files.
✅ Successfully created list file at: /home/taimor/data1/preprocess/minisplit_train_list.json

--- JSON file content (first 5 files) ---
[
  "us-nv-las-vegas-strip_171e5b3acffe59f6.npz",
  "us-pa-pittsburgh-hazelwood_b6a3a678a26552b3.npz",
  "us-nv-las-vegas-strip_54595a44a5045974.npz",
  "us-nv-las-vegas-strip_d2c665d6dcc55511.npz",
  "us-nv-las-vegas-strip_245ecac0995a52b3.npz"
]
------------------------------------------


Got it. Thank you for sharing that!

This is the key piece of information. I see all your 27 .npz files, but there is one crucial file missing: the .json list file.

The train_predictor.py script needs two things:

The path to the folder (--train_set), which you have.

A JSON file (--train_set_list) that lists all the .npz filenames it should use from that folder.

It seems the data_process.py script only creates the .npz files, but not the final list.

No problem! We can create this file ourselves with a simple script.

Step 1: Create the Missing JSON List
Run this Jupyter cell. It will scan your /home/taimor/data1/preprocess folder, find all the .npz files, and write their names into a new file called minisplit_train_list.json.

In [36]:
import os
import fileinput

# --- 1. SET YOUR REPO PATH ---
# This path should be correct based on your logs
diffusion_planner_path = "/home/taimor/V0.1-diffusion-es/Diffusion-Planner"
# -----------------------------

script_path = os.path.join(diffusion_planner_path, "diffusion_planner/utils/data_augmentation.py")
old_text = "torch.concatenate"
new_text = "torch.cat"

print(f"Patching file: {script_path}")
print(f"Replacing '{old_text}' with '{new_text}'")

try:
    patched = False
    with open(script_path, 'r') as f:
        content = f.read()

    if old_text in content:
        new_content = content.replace(old_text, new_text)
        with open(script_path, 'w') as f:
            f.write(new_content)
        print("✅ File patched successfully!")
    else:
        print("File already patched or text not found.")

except Exception as e:
    print(f"❌ An error occurred: {e}")

Patching file: /home/taimor/V0.1-diffusion-es/Diffusion-Planner/diffusion_planner/utils/data_augmentation.py
Replacing 'torch.concatenate' with 'torch.cat'
✅ File patched successfully!


In [37]:
import sys
import os
import subprocess

# --- 1. SET YOUR REPO PATH ---
diffusion_planner_path = "/home/taimor/V0.1-diffusion-es/Diffusion-Planner"
# -----------------------------------


# --- 2. FILENAME WE CREATED ---
TRAIN_SET_LIST_FILENAME = "minisplit_train_list.json"
# -----------------------------------


# --- 3. DEFINE PATHS AND PARAMETERS ---
train_data_path = "/home/taimor/data1/preprocess"
train_set_list_path = os.path.join(train_data_path, TRAIN_SET_LIST_FILENAME)
output_dir = "/home/taimor/data1/diffusion_planner_output"

train_epochs = "50"
batch_size = "16" 
ddp = "False"

os.makedirs(output_dir, exist_ok=True)
print(f"Model checkpoints will be saved to: {output_dir}")
# -------------------------------------


# --- 4. PREPARE AND RUN THE COMMAND ---
script_name = "train_predictor.py"
script_path = os.path.join(diffusion_planner_path, script_name)

if not os.path.exists(script_path):
    print(f"ERROR: Training script not found at '{script_path}'")
elif not os.path.exists(train_set_list_path):
    print(f"ERROR: Training list file not found at '{train_set_list_path}'")
else:
    # Build the full command (Note: --guidance_fn is REMOVED)
    command_list = [
        sys.executable,
        script_name,
        "--train_set", train_data_path,
        "--train_set_list", train_set_list_path,
        "--save_dir", output_dir,
        "--train_epochs", train_epochs,
        "--batch_size", batch_size,
        "--ddp", ddp
    ]

    print("--- Running Training Command ---")
    print(" ".join(command_list))
    print("--------------------------------")
    
    # Run the command
    process = subprocess.Popen(
        command_list,
        cwd=diffusion_planner_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True
    )

    # Print the output in real-time
    for line in process.stdout:
        print(line, end='')

    process.wait()

    print("--------------------------------")
    if process.returncode == 0:
        print("✅ Training script finished successfully!")
    else:
        print(f"❌ Training script failed with return code: {process.returncode}")

Model checkpoints will be saved to: /home/taimor/data1/diffusion_planner_output
--- Running Training Command ---
/home/taimor/miniconda/envs/nuplan/bin/python train_predictor.py --train_set /home/taimor/data1/preprocess --train_set_list /home/taimor/data1/preprocess/minisplit_train_list.json --save_dir /home/taimor/data1/diffusion_planner_output --train_epochs 50 --batch_size 16 --ddp False
--------------------------------
/home/taimor/miniconda/envs/nuplan/lib/python3.9/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.

Training Summary
Goal: Train the Diffusion Planner model using the processed data.

Input: We pointed the train_predictor.py script to the folder containing the .npz files (/home/taimor/data1/preprocess) and the JSON file listing them (minisplit_train_list.json).

Action: The script loaded the 27 .npz samples and trained the model (a type of neural network based on diffusion and transformers) for 50 epochs. It adjusted the model's internal parameters (weights) to learn how to predict future trajectories based on the input data.

Output: The script saved the trained model's weights (as latest.pth, model_epoch_20_..., model_epoch_40_...) and the configuration used during training (args.json) into the /home/taimor/data1/diffusion_planner_output/training_log/... directory. Crucially, this was a very small training run, mainly to test if the code runs without crashing. The resulting model isn't expected to drive well yet.

In [1]:
#run the simulation of the diffusion planner on the minisplit 
!python /home/taimor/V0.1-diffusion-es/nuplan-devkit/nuplan/planning/script/run_simulation.py \
  experiment_name=dp_mini_sanity_2 \
  scenario_builder=nuplan_mini \
  scenario_builder.data_root=/home/taimor/data1/nuplan-v1.1/splits/mini \
  +simulation=closed_loop_nonreactive_agents \
  planner=diffusion_planner \
  planner.diffusion_planner.config.args_file=/home/taimor/V0.1-diffusion-es/Diffusion-Planner/checkpoints/args.json \
  planner.diffusion_planner.ckpt_path=/home/taimor/V0.1-diffusion-es/Diffusion-Planner/checkpoints/model.pth \
  scenario_filter.shuffle=true scenario_filter.limit_total_scenarios=1 \
  worker=sequential verbose=true \
  'hydra.searchpath=[pkg://diffusion_planner.config.scenario_filter,pkg://diffusion_planner.config,pkg://nuplan.planning.script.config.common,pkg://nuplan.planning.script.experiments]'


INFO:nuplan.planning.script.utils:Setting default NUPLAN_EXP_ROOT: /home/taimor/nuplan/exp
/home/taimor/V0.1-diffusion-es/nuplan-devkit/nuplan/planning/script/run_simulation.py:100: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path=CONFIG_PATH, config_name=CONFIG_NAME)
/home/taimor/miniconda/envs/nuplan/lib/python3.9/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'default_simulation': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/home/taimor/miniconda/envs/nuplan/lib/python3.9/site-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  r

In [1]:
#  visualize in nuboard
!python /home/taimor/V0.1-diffusion-es/nuplan-devkit/nuplan/planning/script/run_nuboard.py \
  port_number=7007 \
  worker=sequential \
  simulation_path='["/home/taimor/nuplan/exp/exp/dp_mini_sanity/closed_loop_nonreactive_agents/"]' \
  scenario_builder.data_root=/home/taimor/data1/nuplan-v1.1/splits/mini \
  scenario_builder.map_root=/home/taimor/data1/maps \



INFO:nuplan.planning.script.utils:Setting default NUPLAN_EXP_ROOT: /home/taimor/nuplan/exp
/home/taimor/V0.1-diffusion-es/nuplan-devkit/nuplan/planning/script/run_nuboard.py:67: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path=CONFIG_PATH, config_name=CONFIG_NAME)
/home/taimor/miniconda/envs/nuplan/lib/python3.9/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'default_nuboard': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/home/taimor/miniconda/envs/nuplan/lib/python3.9/site-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = ru